## Control Matching

This notebook creates a **matched control group** for causal analysis using difference-in-differences (DiD).

### Steps:

1. **Load Data**: Imported junior award-winning authors and all award papers with their author information.

2. **Parse Authors**: Extracted all authors (top 5 positions) from award-winning papers, creating a pool of potential controls.

3. **Identify Controls**: Filtered out junior winners to create a control group of senior co-authors and non-winning authors from the same conferences and years.

4. **Nearest Neighbor Matching**: Used 1-to-1 matching to pair each junior winner with a control based on:
    - Conference
    - Author position (first author, second, etc.)
    - Award year
    
    Kept only matches with distance < 1.5 to ensure quality.

5. **Output**: Generated a matched pairs dataset with ~XXX treatment-control pairs, ready for difference-in-differences analysis.

### Key Output:
- `matched_pairs.csv`: Contains matched pairs with author IDs, positions, and match quality metrics
- Position and conference balance statistics to assess matching quality

In [13]:
import pandas as pd
import numpy as np
import requests
import json
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# Load data
juniors = pd.read_csv('B:\\Semester 4 UU\\thesis-best-paper-trajectories\\data\\matched\\junior_authors_all_conferences.csv')
all_awards = pd.read_csv('B:\\Semester 4 UU\\thesis-best-paper-trajectories\\data\\matched\\huang_matched_openalex.csv')

print(f"Juniors: {len(juniors)}")
print(f"Award papers: {len(all_awards)}")

# Parse ALL authors from award papers (juniors + seniors)
def parse_authorships(raw):
    if isinstance(raw, list): return raw
    if pd.isna(raw): return []
    try: return json.loads(raw.replace("'", '"'))
    except: return []

all_authors = []

def parse_authorships(raw):
    if isinstance(raw, list): return raw
    if pd.isna(raw): return []
    try: return json.loads(raw.replace("'", '"'))
    except:
        try: 
            import ast
            return ast.literal_eval(raw)
        except: return []

for _, row in all_awards.iterrows():
    authorships = parse_authorships(row['authorships'])
    for pos, a in enumerate(authorships[:5], 1):  # top 5 only
        author = a.get('author', {})
        all_authors.append({
            'author_id'      : author.get('id'),
            'author_name'    : author.get('display_name'),
            'conference'     : row['conference'],
            'award_year'     : row['year'],
            'author_position': pos,
            'match_score'    : row['match_route'],
        })

all_df = pd.DataFrame(all_authors)
controls = all_df[~all_df['author_id'].isin(juniors['author_id'])]

print(f"All authors:  {len(all_df)}")
print(f"Controls:     {len(controls)}")
print(f"Juniors:      {len(juniors)}")


Juniors: 603
Award papers: 912
All authors:  2964
Controls:     2330
Juniors:      603


In [15]:
# Categorical encoding for matching
features = ['conference', 'author_position', 'award_year']
feature_map = {feat: pd.Categorical(all_df[feat]).codes for feat in features}

treat_rows = juniors.index.to_numpy()
ctrl_rows  = controls.index.to_numpy()
treat_feat = np.column_stack([feature_map[f][treat_rows] for f in features])
ctrl_feat  = np.column_stack([feature_map[f][ctrl_rows] for f in features])


nn = NearestNeighbors(n_neighbors=1, metric='euclidean')
nn.fit(ctrl_feat)
dists, ctrl_idxs = nn.kneighbors(treat_feat)

# Keep good matches only
good = dists[:,0] < 1.5
matched_pairs = []
for i in np.where(good)[0]:
    treat_idx = juniors.index[i]
    ctrl_idx  = controls.index[ctrl_idxs[i,0]]
    
    matched_pairs.append({
        'treated_id'    : juniors.iloc[treat_idx]['author_id'],
        'treated_name'  : juniors.iloc[treat_idx]['author_name'],
        'control_id'    : controls.iloc[ctrl_idx]['author_id'],
        'control_name'  : controls.iloc[ctrl_idx]['author_name'],
        'conference'    : juniors.iloc[treat_idx]['conference'],
        'award_year'    : juniors.iloc[treat_idx]['award_year'],
        'treat_pos'     : juniors.iloc[treat_idx]['author_position'],
        'ctrl_pos'      : controls.iloc[ctrl_idx]['author_position'],
        'match_dist'    : float(dists[i,0]),
    })

matched = pd.DataFrame(matched_pairs)
print(f"✅ Matched {len(matched)} pairs out of {len(juniors)} juniors")
print(f"Avg distance: {matched['match_dist'].mean():.2f}")


✅ Matched 603 pairs out of 603 juniors
Avg distance: 0.04


In [16]:
matched.to_csv('B:\\Semester 4 UU\\thesis-best-paper-trajectories\\data\\matched\\matched_pairs.csv', index=False)

print("Position balance:")
print(pd.crosstab(matched['treat_pos'], matched['ctrl_pos']))

print("\nConference overlap (top 10):")
print(matched['conference'].value_counts().head(10))

print("\n✅ Ready for DiD analysis!")


Position balance:
ctrl_pos    1   2   3   4   5
treat_pos                    
1          60  88  67  38  38
2          46  50  46  18  33
3          15  42  32  12  18

Conference overlap (top 10):
conference
CHI     119
ICSE     81
FSE      47
UIST     27
PLDI     26
OSDI     22
SOSP     22
ACL      19
VLDB     18
AAAI     16
Name: count, dtype: int64

✅ Ready for DiD analysis!
